# [CNN Lab] - Building Neural Networks with nn.Module & CNNs

In this notebook, you'll learn the fundamental building blocks of PyTorch neural networks by implementing your own custom modules and building Convolutional Neural Networks (CNNs) from scratch.

This lab is designed to teach you:

1. **Custom modules**: How to create your own layers by subclassing `nn.Module`
2. **Building blocks**: Implementing ReLU activations and Linear layers
3. **CNNs from scratch**: Understanding and implementing convolutional architectures
4. **Batch Normalization**: How and why it stabilizes training
5. **Real-world application**: Training a CNN on CIFAR-10 image classification

## How to Use This Notebook

This notebook builds on concepts from the previous MLP lab:

- **Read carefully**: Each section introduces new concepts progressively
- **Run linearly**: Execute cells in order from top to bottom
- **Complete exercises**: Implement the TODOs marked in code cells
- **Check your understanding**: Use the questions and checkpoints
- **Experiment**: After completing exercises, try modifying architectures

Expected time: **~3 hours**

## Content & Learning Objectives

This lab is divided into 4 main sections:

### 1️⃣ Understanding nn.Module
Learn how to create custom PyTorch modules by implementing your own layers.

> ##### Learning Objectives
> 
> - Understand the `nn.Module` base class and its methods
> - Learn about `nn.Parameter` for trainable weights
> - Implement custom `ReLU` and `Linear` modules
> - Build and train a simple MLP using custom modules

### 2️⃣ Introduction to Convolutional Neural Networks
Understand what convolutions are and why they're powerful for image processing.

> ##### Learning Objectives
>
> - Understand convolution operations and their parameters
> - Learn about receptive fields and spatial hierarchies
> - Understand pooling operations
> - Learn how CNNs extract hierarchical features

### 3️⃣ Batch Normalization
Learn how batch normalization stabilizes and accelerates training.

> ##### Learning Objectives
>
> - Understand the internal covariate shift problem
> - Learn how batch normalization works
> - Understand the difference between train and eval modes
> - Apply batch normalization in CNNs

### 4️⃣ Building and Training CNNs
Implement a complete CNN architecture and train it on CIFAR-10.

> ##### Learning Objectives
>
> - Implement a reusable CNN block module
> - Build a complete CNN architecture
> - Train on CIFAR-10 dataset
> - Compare CNN performance with MLPs

## Setup code

In [ ]:
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from typing import Optional

import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms

In [ ]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


set_seed(42)

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print(f"Using device: {device}")

# 1️⃣ Understanding nn.Module

## What is nn.Module?

`nn.Module` is the base class for all neural network modules in PyTorch. Everything from a simple ReLU activation to a complete ResNet model is an `nn.Module`.

**Why is nn.Module important?**

1. **Automatic parameter tracking**: When you define parameters in `__init__`, PyTorch automatically tracks them
2. **Easy device management**: Moving a module to GPU/CPU moves all its parameters
3. **Training mode management**: Easily switch between train and eval modes
4. **Composability**: Modules can contain other modules, making complex architectures simple

## The nn.Module Pattern

Every `nn.Module` subclass follows this pattern:

```python
class MyModule(nn.Module):
    def __init__(self, ...parameters...):
        super().__init__()
        # Define layers and parameters here
        self.param1 = nn.Parameter(torch.randn(...))
        self.layer1 = nn.Linear(...)
    
    def forward(self, x):
        # Define the forward pass
        x = self.layer1(x)
        return x
```

**Key components:**

- **`__init__`**: Define your layers and parameters
- **`super().__init__()`**: MUST call the parent class constructor first
- **`forward`**: Define how data flows through the module
- **No `backward`**: PyTorch autograd handles this automatically!

## nn.Parameter: Trainable Weights

When you want a tensor to be **trainable** (updated by the optimizer), wrap it in `nn.Parameter`:

```python
self.weight = nn.Parameter(torch.randn(out_features, in_features))
self.bias = nn.Parameter(torch.zeros(out_features))
```

**What nn.Parameter does:**
- Registers the tensor as a module parameter
- Makes it visible to `model.parameters()`
- Enables automatic gradient computation
- Moves with the model when you call `.to(device)`

**Regular tensor vs Parameter:**
```python
self.regular = torch.randn(10, 10)      # NOT trainable, NOT tracked
self.param = nn.Parameter(torch.randn(10, 10))  # Trainable, tracked
```

## Exercise - Implement ReLU

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Importance: 🔵🔵🔵🔵⚪
>
> You should spend up to 10 minutes on this exercise.
> ```

Let's start simple: implement the ReLU (Rectified Linear Unit) activation function.

**ReLU formula**: $\text{ReLU}(x) = \max(0, x)$

**Why ReLU?**
- Simple and fast to compute
- Helps prevent vanishing gradients
- Most popular activation for hidden layers

**Your task:**
1. Subclass `nn.Module`
2. Implement the `forward` method
3. Use `torch.maximum` or `torch.clamp` to implement ReLU

**Note**: ReLU has no trainable parameters, so you don't need `__init__` beyond calling `super().__init__()`!

<details>
<summary>Hint</summary>

```python
class ReLU(nn.Module):
    def forward(self, x):
        return torch.maximum(x, torch.zeros_like(x))
        # OR: return torch.clamp(x, min=0)
        # OR: return F.relu(x)  # using PyTorch's implementation
```
</details>

In [ ]:
class ReLU(nn.Module):
    def forward(self, x):  # TODO: Implement ReLU activation
        ...

In [ ]:
# Test your ReLU implementation
relu = ReLU()
test_input = torch.tensor([-2.0, -1.0, 0.0, 1.0, 2.0])
output = relu(test_input)

print(f"Input:  {test_input}")
print(f"Output: {output}")
print(f"Expected: tensor([0., 0., 0., 1., 2.])")

assert torch.allclose(output, torch.tensor([0., 0., 0., 1., 2.])), "ReLU implementation is incorrect!"
print("\n✅ ReLU implementation correct!")

## Exercise - Implement Linear

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Importance: 🔵🔵🔵🔵🔵
>
> You should spend up to 15-20 minutes on this exercise.
> ```

Now implement a Linear (fully connected) layer. This is the fundamental building block of neural networks.

**Linear layer formula**: $y = xW^T + b$

where:
- $x$ is the input: shape `(batch_size, in_features)`
- $W$ is the weight matrix: shape `(out_features, in_features)`
- $b$ is the bias vector: shape `(out_features,)`
- $y$ is the output: shape `(batch_size, out_features)`

**Your task:**
1. Define `weight` and `bias` as `nn.Parameter` in `__init__`
2. Initialize weights using `torch.randn` (or Kaiming initialization)
3. Initialize bias as zeros using `torch.zeros`
4. Implement the forward pass using `torch.matmul` or `@`

**Important notes:**
- The weight shape is `(out_features, in_features)`, not the other way around!
- You need to transpose the weight matrix in the forward pass
- Use `@` or `torch.matmul` for matrix multiplication

<details>
<summary>Hint - Initialization</summary>

For better training, you can use Kaiming (He) initialization:

```python
nn.init.kaiming_uniform_(self.weight, a=np.sqrt(5))
```

But simple random initialization works fine for this exercise:
```python
self.weight = nn.Parameter(torch.randn(out_features, in_features) * 0.01)
```
</details>

<details>
<summary>Hint - Forward pass</summary>

```python
def forward(self, x):
    return x @ self.weight.T + self.bias
    # OR: return torch.matmul(x, self.weight.T) + self.bias
```
</details>

In [ ]:
class Linear(nn.Module):
    def __init__(self, in_features: int, out_features: int):
        super().__init__()
        # TODO: Initialize weight and bias as nn.Parameter
        self.weight = ...
        self.bias = ...
    
    def forward(self, x):  # TODO: Implement forward pass
        ...
    
    def extra_repr(self):
        return f"in_features={self.weight.shape[1]}, out_features={self.weight.shape[0]}"

In [ ]:
# Test your Linear implementation
linear = Linear(in_features=5, out_features=3)

# Test 1: Check parameter shapes
print("Weight shape:", linear.weight.shape)  # Should be (3, 5)
print("Bias shape:", linear.bias.shape)      # Should be (3,)
assert linear.weight.shape == (3, 5), "Weight shape is incorrect!"
assert linear.bias.shape == (3,), "Bias shape is incorrect!"

# Test 2: Check forward pass
test_input = torch.randn(2, 5)  # batch_size=2, in_features=5
output = linear(test_input)
print("\nInput shape:", test_input.shape)   # (2, 5)
print("Output shape:", output.shape)       # Should be (2, 3)
assert output.shape == (2, 3), "Output shape is incorrect!"

# Test 3: Check parameters are registered
num_params = sum(p.numel() for p in linear.parameters())
print(f"\nTotal parameters: {num_params}")  # Should be 5*3 + 3 = 18
assert num_params == 18, "Parameters not correctly registered!"

print("\n✅ Linear implementation correct!")
print(f"\nModule representation:\n{linear}")

## Exercise - Build a Simple MLP

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Importance: 🔵🔵🔵🔵🔵
>
> You should spend up to 15-20 minutes on this exercise.
> ```

Now let's combine your custom `Linear` and `ReLU` modules to build a complete Multi-Layer Perceptron!

**Architecture:**
```
Input (784) → Linear(784, 128) → ReLU → Linear(128, 64) → ReLU → Linear(64, 10) → Output
```

**Your task:**
1. Create three `Linear` layers with the specified sizes
2. Create two `ReLU` activations
3. Implement the forward pass by chaining these modules
4. Add a `Flatten` layer at the beginning to handle image inputs

**Note**: We don't apply ReLU after the final layer because we'll use `CrossEntropyLoss`, which expects raw logits.

<details>
<summary>Hint</summary>

```python
class SimpleMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.fc1 = Linear(784, 128)
        self.relu1 = ReLU()
        self.fc2 = Linear(128, 64)
        self.relu2 = ReLU()
        self.fc3 = Linear(64, 10)
    
    def forward(self, x):
        x = self.flatten(x)
        x = self.relu1(self.fc1(x))
        x = self.relu2(self.fc2(x))
        x = self.fc3(x)
        return x
```
</details>

In [ ]:
class SimpleMLP(nn.Module):
    def __init__(self):
        super().__init__()
        # TODO: Define layers
        self.flatten = ...
        self.fc1 = ...
        self.relu1 = ...
        self.fc2 = ...
        self.relu2 = ...
        self.fc3 = ...
    
    def forward(self, x):  # TODO: Implement forward pass
        ...

In [ ]:
# Test your MLP implementation
model = SimpleMLP().to(device)

# Test with MNIST-like input
test_input = torch.randn(4, 1, 28, 28).to(device)  # batch_size=4
output = model(test_input)

print(f"Input shape: {test_input.shape}")
print(f"Output shape: {output.shape}")
assert output.shape == (4, 10), "Output shape is incorrect!"

# Count parameters
num_params = sum(p.numel() for p in model.parameters())
print(f"\nTotal parameters: {num_params:,}")
# 784*128 + 128 + 128*64 + 64 + 64*10 + 10 = 109,386
assert num_params == 109386, f"Expected 109,386 parameters, got {num_params}"

print("\n✅ SimpleMLP implementation correct!")
print(f"\nModel architecture:\n{model}")

## Training the MLP on MNIST

Let's verify your custom modules work correctly by training on MNIST. We'll reuse the training utilities from the previous lab.

In [ ]:
# Load MNIST dataset
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

train_dataset = datasets.MNIST(root="./data", train=True, download=True, transform=transform)
test_dataset = datasets.MNIST(root="./data", train=False, download=True, transform=transform)

# Create smaller subset for quick training
train_subset = torch.utils.data.Subset(train_dataset, range(10000))

train_loader = DataLoader(train_subset, batch_size=128, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=256, shuffle=False)

print(f"Training samples: {len(train_subset):,}")
print(f"Test samples: {len(test_dataset):,}")

In [ ]:
def train_one_epoch(model, dataloader, criterion, optimizer, device):
    model.train()
    total_loss = 0.0
    total_correct = 0
    total_examples = 0
    
    for X_batch, y_batch in dataloader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        
        optimizer.zero_grad()
        logits = model(X_batch)
        loss = criterion(logits, y_batch)
        loss.backward()
        optimizer.step()
        
        batch_size = y_batch.size(0)
        total_loss += loss.item() * batch_size
        total_correct += (logits.argmax(dim=1) == y_batch).sum().item()
        total_examples += batch_size
    
    return total_loss / total_examples, total_correct / total_examples

def evaluate(model, dataloader, criterion, device):
    model.eval()
    total_loss = 0.0
    total_correct = 0
    total_examples = 0
    
    with torch.no_grad():
        for X_batch, y_batch in dataloader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            
            logits = model(X_batch)
            loss = criterion(logits, y_batch)
            
            batch_size = y_batch.size(0)
            total_loss += loss.item() * batch_size
            total_correct += (logits.argmax(dim=1) == y_batch).sum().item()
            total_examples += batch_size
    
    return total_loss / total_examples, total_correct / total_examples

In [ ]:
# Train the custom MLP
set_seed(42)
model = SimpleMLP().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

print("Training SimpleMLP on MNIST...\n")
for epoch in range(1, 4):
    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
    test_loss, test_acc = evaluate(model, test_loader, criterion, device)
    
    print(f"Epoch {epoch}/3 | "
          f"train_loss={train_loss:.4f}, train_acc={train_acc:.4f} | "
          f"test_loss={test_loss:.4f}, test_acc={test_acc:.4f}")

print("\n✅ Custom MLP trains successfully!")

# 2️⃣ Introduction to Convolutional Neural Networks

## Why CNNs for Images?

MLPs work for MNIST, but they have serious limitations for images:

**Problems with MLPs for images:**
1. **Too many parameters**: A 224×224 RGB image has 150,528 pixels. A single hidden layer with 1000 units would need 150 million parameters!
2. **No spatial awareness**: MLPs treat each pixel independently, ignoring that nearby pixels are related
3. **Not translation invariant**: An MLP trained on centered faces won't recognize shifted faces
4. **Can't handle different input sizes**: MLPs need fixed-size inputs

**CNNs solve these problems by:**
1. **Parameter sharing**: Use the same filter across the entire image
2. **Local connectivity**: Each neuron only looks at a small region
3. **Translation equivariance**: If input shifts, output shifts correspondingly
4. **Hierarchical features**: Early layers detect edges, later layers detect complex patterns

## What is a Convolution?

A **convolution** slides a small filter (kernel) across an image, computing a weighted sum at each position.

### Key Parameters

**1. Kernel size**: The size of the sliding window (commonly 3×3, 5×5, or 7×7)
- Larger kernels see more context but have more parameters
- Modern CNNs mostly use 3×3 kernels

**2. Stride**: How many pixels to move the filter each step
- Stride=1: Move one pixel at a time (most common)
- Stride=2: Move two pixels, reducing output size by half

**3. Padding**: Adding zeros around the input border
- `padding=0`: No padding, output size shrinks
- `padding='same'`: Pad so output size equals input size (when stride=1)
- `padding=k//2`: Common choice for kernel size k

**4. Number of channels**:
- **Input channels**: RGB image has 3 input channels
- **Output channels (filters)**: How many different features to detect

### Output Size Formula

For an input of size $(H, W)$ with kernel size $K$, padding $P$, and stride $S$:

$$H_{\text{out}} = \left\lfloor \frac{H + 2P - K}{S} \right\rfloor + 1$$

$$W_{\text{out}} = \left\lfloor \frac{W + 2P - K}{S} \right\rfloor + 1$$

### Example

```python
conv = nn.Conv2d(
    in_channels=3,      # RGB input
    out_channels=64,    # 64 different filters
    kernel_size=3,      # 3×3 filter
    stride=1,           # Move 1 pixel at a time
    padding=1           # Pad to maintain size
)
```

Input: `(batch, 3, 32, 32)` → Output: `(batch, 64, 32, 32)`

## Pooling Layers

**Pooling** reduces spatial dimensions while keeping important features.

### MaxPool2d

Takes the maximum value in each window:

```
Input (4×4):        MaxPool2d(2×2, stride=2):       Output (2×2):
[1  2  3  4]                                        [6  8]
[5  6  7  8]        →  Take max in each 2×2  →     [14 16]
[9  10 11 12]          window
[13 14 15 16]
```

**Benefits:**
- Reduces computation for subsequent layers
- Provides translation invariance (small shifts don't change output)
- Increases receptive field of later layers
- Helps prevent overfitting

**Common configuration**: `kernel_size=2, stride=2` (halves spatial dimensions)

```python
pool = nn.MaxPool2d(kernel_size=2, stride=2)
```

Input: `(batch, channels, 32, 32)` → Output: `(batch, channels, 16, 16)`

# 3️⃣ Batch Normalization

## The Internal Covariate Shift Problem

During training, the distribution of inputs to each layer keeps changing as the previous layers' parameters update. This is called **internal covariate shift**.

**Why is this a problem?**
- Each layer must constantly adapt to new input distributions
- Slows down training
- Requires careful initialization and small learning rates
- Makes training deep networks difficult

## How Batch Normalization Works

Batch Normalization (BatchNorm) normalizes the inputs to each layer within a mini-batch.

### The BatchNorm Formula

For a mini-batch of activations $x_1, x_2, ..., x_m$:

**1. Compute mean and variance:**
$$\mu_B = \frac{1}{m} \sum_{i=1}^{m} x_i$$
$$\sigma_B^2 = \frac{1}{m} \sum_{i=1}^{m} (x_i - \mu_B)^2$$

**2. Normalize:**
$$\hat{x}_i = \frac{x_i - \mu_B}{\sqrt{\sigma_B^2 + \epsilon}}$$

where $\epsilon$ is a small constant (e.g., 1e-5) for numerical stability.

**3. Scale and shift (learnable parameters):**
$$y_i = \gamma \hat{x}_i + \beta$$

where $\gamma$ (scale) and $\beta$ (shift) are learned parameters.

### Why Scale and Shift?

The $\gamma$ and $\beta$ parameters allow the network to **undo** the normalization if needed. This is important because sometimes the optimal distribution isn't zero-mean and unit variance.

For example, if $\gamma = \sqrt{\sigma_B^2}$ and $\beta = \mu_B$, the layer can recover the original activations!

## Train vs Eval Mode

BatchNorm behaves differently during training and evaluation:

**Training mode** (`model.train()`):
- Uses mini-batch statistics ($\mu_B$, $\sigma_B^2$)
- Updates running mean and variance using exponential moving average
- Adds noise (different batches have different statistics)

**Eval mode** (`model.eval()`):
- Uses running mean and variance (accumulated during training)
- Deterministic (same input always gives same output)
- No batch statistics needed (can process single samples)

### Running Statistics

During training, BatchNorm tracks running estimates:

$$\text{running\_mean} \leftarrow (1 - \text{momentum}) \cdot \text{running\_mean} + \text{momentum} \cdot \mu_B$$

These running statistics are used during evaluation.

## Benefits of Batch Normalization

1. **Faster training**: Can use higher learning rates
2. **Better gradients**: Prevents gradients from exploding or vanishing
3. **Less sensitive to initialization**: Network trains well with various initializations
4. **Regularization effect**: The noise from batch statistics acts as regularization
5. **Better generalization**: Often improves test accuracy

## Where to Place BatchNorm?

Two common patterns:

**Pattern 1** (original paper): Conv → BatchNorm → ReLU
```python
nn.Conv2d(...)
nn.BatchNorm2d(...)
nn.ReLU()
```

**Pattern 2** (modern): Conv → ReLU → BatchNorm
```python
nn.Conv2d(...)
nn.ReLU()
nn.BatchNorm2d(...)
```

Both work well in practice. We'll use Pattern 1 in this lab.

## BatchNorm2d for CNNs

For 2D convolutions, we use `nn.BatchNorm2d`:

```python
conv = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding=1)
bn = nn.BatchNorm2d(num_features=64)  # Matches out_channels of conv
relu = nn.ReLU()
```

The `num_features` parameter must match the number of channels (out_channels of the preceding conv layer).

# 4️⃣ Building and Training CNNs

Now we'll implement a modular CNN architecture using the building blocks we've learned.

## The CNN Block Pattern

Modern CNNs are built from repeated blocks. A typical CNN block consists of:

```
Conv2d → BatchNorm2d → ReLU → MaxPool2d
```

By making this a reusable module, we can easily build deep networks.

## Exercise - Implement CNN_Block

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Importance: 🔵🔵🔵🔵🔵
>
> You should spend up to 15-20 minutes on this exercise.
> ```

Implement a reusable CNN block that combines convolution, batch normalization, activation, and pooling.

**Your task:**
1. Add a `Conv2d` layer with `kernel_size=3` and `padding=1`
2. Add a `BatchNorm2d` layer (num_features = out_channels)
3. Add a `ReLU` activation
4. Optionally add `MaxPool2d` (only if `use_pool=True`)

**Parameters:**
- `in_channels`: Number of input channels
- `out_channels`: Number of output channels (number of filters)
- `use_pool`: Whether to include max pooling (default: True)

**Hints:**
- Use `kernel_size=3, padding=1` for Conv2d to maintain spatial dimensions
- Use `kernel_size=2, stride=2` for MaxPool2d to halve spatial dimensions
- Remember to call `super().__init__()`!

<details>
<summary>Hint - Implementation structure</summary>

```python
class CNN_Block(nn.Module):
    def __init__(self, in_channels, out_channels, use_pool=True):
        super().__init__()
        self.conv = nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1)
        self.bn = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU()
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2) if use_pool else nn.Identity()
    
    def forward(self, x):
        x = self.conv(x)
        x = self.bn(x)
        x = self.relu(x)
        x = self.pool(x)
        return x
```

Note: `nn.Identity()` is a no-op layer that returns its input unchanged.
</details>

In [ ]:
class CNN_Block(nn.Module):
    def __init__(self, in_channels: int, out_channels: int, use_pool: bool = True):
        super().__init__()
        # TODO: Implement CNN block layers
        self.conv = ...
        self.bn = ...
        self.relu = ...
        self.pool = ...
    
    def forward(self, x):  # TODO: Implement forward pass
        ...

In [ ]:
# Test CNN_Block
block = CNN_Block(in_channels=3, out_channels=32, use_pool=True).to(device)

# Test with CIFAR-10 sized input
test_input = torch.randn(4, 3, 32, 32).to(device)  # (batch, channels, height, width)
output = block(test_input)

print(f"Input shape:  {test_input.shape}")
print(f"Output shape: {output.shape}")
print(f"Expected:     torch.Size([4, 32, 16, 16])")

# With pooling: 32×32 → 16×16
assert output.shape == (4, 32, 16, 16), "Output shape incorrect with pooling!"

# Test without pooling
block_no_pool = CNN_Block(in_channels=3, out_channels=32, use_pool=False).to(device)
output_no_pool = block_no_pool(test_input)
assert output_no_pool.shape == (4, 32, 32, 32), "Output shape incorrect without pooling!"

print("\n✅ CNN_Block implementation correct!")
print(f"\nBlock with pooling:\n{block}")

## Exercise - Build a Complete CNN for CIFAR-10

> ```yaml
> Difficulty: 🔴🔴🔴🔴⚪
> Importance: 🔵🔵🔵🔵🔵
>
> You should spend up to 20-30 minutes on this exercise.
> ```

Now let's build a complete CNN for CIFAR-10 classification using your CNN_Block!

**CIFAR-10 dataset:**
- 60,000 color images (32×32 pixels, 3 channels)
- 10 classes: airplane, automobile, bird, cat, deer, dog, frog, horse, ship, truck
- More challenging than MNIST!

**Architecture:**
```
Input (3, 32, 32)
    ↓
CNN_Block(3 → 32)    [Output: (32, 16, 16)]
    ↓
CNN_Block(32 → 64)   [Output: (64, 8, 8)]
    ↓
CNN_Block(64 → 128)  [Output: (128, 4, 4)]
    ↓
Flatten              [Output: (2048,)]
    ↓
Linear(2048 → 256)
    ↓
ReLU
    ↓
Dropout(0.5)
    ↓
Linear(256 → 10)     [Output: (10,)]
```

**Your task:**
1. Create three CNN_Block layers with the specified channel sizes
2. Add a Flatten layer to convert 4D tensors to 2D
3. Add two fully connected layers (2048→256→10)
4. Add ReLU activation and Dropout after the first FC layer
5. Implement the forward pass

**Important notes:**
- Each CNN_Block with pooling halves the spatial dimensions: 32→16→8→4
- After the last block: 128 channels × 4 × 4 = 2048 features
- Dropout helps prevent overfitting (use p=0.5)
- No activation after the final layer (CrossEntropyLoss expects logits)

<details>
<summary>Hint</summary>

```python
class CIFAR_CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.block1 = CNN_Block(3, 32)
        self.block2 = CNN_Block(32, 64)
        self.block3 = CNN_Block(64, 128)
        
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(128 * 4 * 4, 256)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.5)
        self.fc2 = nn.Linear(256, 10)
    
    def forward(self, x):
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        x = self.flatten(x)
        x = self.fc1(x)
        x = self.relu(x)
        x = self.dropout(x)
        x = self.fc2(x)
        return x
```
</details>

In [ ]:
class CIFAR_CNN(nn.Module):
    def __init__(self):
        super().__init__()
        # TODO: Implement CNN architecture
        self.block1 = ...
        self.block2 = ...
        self.block3 = ...
        
        self.flatten = ...
        self.fc1 = ...
        self.relu = ...
        self.dropout = ...
        self.fc2 = ...
    
    def forward(self, x):  # TODO: Implement forward pass
        ...

In [ ]:
# Test CIFAR_CNN
model = CIFAR_CNN().to(device)

# Test with CIFAR-10 input
test_input = torch.randn(8, 3, 32, 32).to(device)
output = model(test_input)

print(f"Input shape:  {test_input.shape}")
print(f"Output shape: {output.shape}")
assert output.shape == (8, 10), "Output shape is incorrect!"

# Count parameters
num_params = sum(p.numel() for p in model.parameters())
print(f"\nTotal parameters: {num_params:,}")

print("\n✅ CIFAR_CNN implementation correct!")
print(f"\nModel architecture:\n{model}")

## Loading CIFAR-10 Dataset

Let's load and prepare the CIFAR-10 dataset for training.

In [ ]:
# CIFAR-10 normalization values (computed from training set)
CIFAR_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR_STD = (0.2470, 0.2435, 0.2616)

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(CIFAR_MEAN, CIFAR_STD)
])

# Load datasets
train_dataset = datasets.CIFAR10(root="./data", train=True, download=True, transform=transform)
test_dataset = datasets.CIFAR10(root="./data", train=False, download=True, transform=transform)

# Create train/validation split
train_size = 45000
val_size = 5000
train_subset, val_subset = random_split(
    train_dataset,
    [train_size, val_size],
    generator=torch.Generator().manual_seed(42)
)

# Create dataloaders
BATCH_SIZE = 128
train_loader = DataLoader(train_subset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_subset, batch_size=256, shuffle=False, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=256, shuffle=False, num_workers=2)

print(f"Training samples: {len(train_subset):,}")
print(f"Validation samples: {len(val_subset):,}")
print(f"Test samples: {len(test_dataset):,}")

# Class names
classes = ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']

In [ ]:
# Visualize some CIFAR-10 samples
def denormalize(img):
    """Undo normalization for visualization"""
    img = img * torch.tensor(CIFAR_STD).view(3, 1, 1) + torch.tensor(CIFAR_MEAN).view(3, 1, 1)
    return img.clamp(0, 1)

images, labels = next(iter(train_loader))

fig, axes = plt.subplots(2, 8, figsize=(12, 3))
axes = axes.ravel()
for i in range(16):
    img = denormalize(images[i])
    img = img.permute(1, 2, 0).numpy()  # CHW -> HWC for matplotlib
    axes[i].imshow(img)
    axes[i].set_title(classes[labels[i]], fontsize=8)
    axes[i].axis('off')

plt.tight_layout()
plt.show()

## Training the CNN on CIFAR-10

Now let's train our CNN! We'll use the same training utilities as before.

In [ ]:
def train_model(model, train_loader, val_loader, criterion, optimizer, epochs=10, device=device):
    history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
    
    for epoch in range(1, epochs + 1):
        # Training
        model.train()
        train_loss = 0.0
        train_correct = 0
        train_total = 0
        
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            
            optimizer.zero_grad()
            logits = model(X_batch)
            loss = criterion(logits, y_batch)
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item() * y_batch.size(0)
            train_correct += (logits.argmax(dim=1) == y_batch).sum().item()
            train_total += y_batch.size(0)
        
        train_loss /= train_total
        train_acc = train_correct / train_total
        
        # Validation
        model.eval()
        val_loss = 0.0
        val_correct = 0
        val_total = 0
        
        with torch.no_grad():
            for X_batch, y_batch in val_loader:
                X_batch, y_batch = X_batch.to(device), y_batch.to(device)
                
                logits = model(X_batch)
                loss = criterion(logits, y_batch)
                
                val_loss += loss.item() * y_batch.size(0)
                val_correct += (logits.argmax(dim=1) == y_batch).sum().item()
                val_total += y_batch.size(0)
        
        val_loss /= val_total
        val_acc = val_correct / val_total
        
        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)
        
        print(f"Epoch {epoch:02d}/{epochs} | "
              f"train_loss={train_loss:.4f}, train_acc={train_acc:.4f} | "
              f"val_loss={val_loss:.4f}, val_acc={val_acc:.4f}")
    
    return history

In [ ]:
# Initialize and train the model
set_seed(42)
model = CIFAR_CNN().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

print("Training CIFAR_CNN on CIFAR-10...\n")
print(f"Model has {sum(p.numel() for p in model.parameters()):,} parameters\n")

history = train_model(
    model,
    train_loader,
    val_loader,
    criterion,
    optimizer,
    epochs=15,
    device=device
)

In [ ]:
# Plot training curves
def plot_history(history):
    epochs = np.arange(1, len(history["train_loss"]) + 1)
    
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    
    axes[0].plot(epochs, history["train_loss"], marker="o", label="Train")
    axes[0].plot(epochs, history["val_loss"], marker="o", label="Validation")
    axes[0].set_title("Loss")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Cross-entropy")
    axes[0].grid(alpha=0.3)
    axes[0].legend()
    
    axes[1].plot(epochs, history["train_acc"], marker="o", label="Train")
    axes[1].plot(epochs, history["val_acc"], marker="o", label="Validation")
    axes[1].set_title("Accuracy")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Accuracy")
    axes[1].grid(alpha=0.3)
    axes[1].legend()
    
    plt.tight_layout()
    plt.show()

plot_history(history)

## Final Evaluation on Test Set

In [ ]:
# Evaluate on test set
model.eval()
test_correct = 0
test_total = 0

with torch.no_grad():
    for X_batch, y_batch in test_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        logits = model(X_batch)
        test_correct += (logits.argmax(dim=1) == y_batch).sum().item()
        test_total += y_batch.size(0)

test_acc = test_correct / test_total
print(f"\nFinal Test Accuracy: {test_acc:.4f} ({test_acc*100:.2f}%)")

## Visualizing Predictions

In [ ]:
# Visualize some predictions
model.eval()
images, labels = next(iter(test_loader))
images_display = images[:16]
labels_display = labels[:16]

with torch.no_grad():
    logits = model(images_display.to(device))
    predictions = logits.argmax(dim=1).cpu()

fig, axes = plt.subplots(2, 8, figsize=(14, 4))
axes = axes.ravel()

for i in range(16):
    img = denormalize(images_display[i])
    img = img.permute(1, 2, 0).numpy()
    axes[i].imshow(img)
    
    true_label = classes[labels_display[i]]
    pred_label = classes[predictions[i]]
    color = 'green' if predictions[i] == labels_display[i] else 'red'
    
    axes[i].set_title(f"T: {true_label}\nP: {pred_label}", fontsize=8, color=color)
    axes[i].axis('off')

plt.tight_layout()
plt.show()

correct = (predictions == labels_display).sum().item()
print(f"\nCorrect predictions in this batch: {correct}/16")

# Summary & Key Takeaways

## What We've Learned

### 1. nn.Module Pattern
- **`__init__`**: Define your layers and parameters
- **`forward`**: Define how data flows through the module
- **`nn.Parameter`**: Makes tensors trainable and tracked
- **Composability**: Modules can contain other modules

### 2. Convolutional Neural Networks
- **Local connectivity**: Each neuron only looks at a small region
- **Parameter sharing**: Same filter used across the entire image
- **Translation equivariance**: Shifting input shifts output correspondingly
- **Hierarchical features**: Early layers detect edges, later layers detect complex patterns

### 3. Batch Normalization
- **Normalizes activations**: Zero mean and unit variance within each mini-batch
- **Learnable parameters**: $\gamma$ (scale) and $\beta$ (shift)
- **Different modes**: Uses batch statistics during training, running statistics during evaluation
- **Benefits**: Faster training, better gradients, less sensitive to initialization

### 4. Building Block Pattern
- **Modular design**: Conv → BatchNorm → ReLU → Pool as a reusable block
- **Easy to experiment**: Change block configuration without rewriting entire network
- **Scalable**: Stack blocks to create deeper networks

## CNN vs MLP Performance

**On CIFAR-10:**
- **MLP**: ~50-55% accuracy (barely better than random)
- **CNN**: ~70-75% accuracy (our implementation)
- **State-of-the-art CNNs**: >95% accuracy

**Why CNNs win:**
1. Fewer parameters (parameter sharing)
2. Spatial structure awareness
3. Translation invariance
4. Hierarchical feature learning

## Best Practices

✅ **Do:**
- Use `nn.Module` for all custom layers
- Call `super().__init__()` first in `__init__`
- Use `nn.Parameter` for trainable weights
- Add batch normalization after convolutions
- Use 3×3 kernels with padding=1 (modern standard)
- Call `model.eval()` before evaluation
- Normalize your input data

❌ **Don't:**
- Forget to call `super().__init__()`
- Use very large kernels (7×7+) unless necessary
- Apply activation after the final layer (let loss handle it)
- Mix up channel dimensions
- Forget about the flattening step before fully connected layers

## Next Steps

To further improve your CNN:

1. **Data augmentation**: RandomCrop, RandomHorizontalFlip, etc.
2. **Learning rate scheduling**: Reduce LR when validation plateaus
3. **More layers**: Deeper networks often perform better
4. **Residual connections**: Skip connections help train very deep networks
5. **Different architectures**: Try VGG, ResNet, DenseNet patterns
6. **Hyperparameter tuning**: Experiment with learning rates, batch sizes, dropout rates

## Congratulations!

You've successfully:
- ✅ Implemented custom PyTorch modules
- ✅ Built a Multi-Layer Perceptron from scratch
- ✅ Understood convolutional operations
- ✅ Learned about batch normalization
- ✅ Created a modular CNN architecture
- ✅ Trained a CNN on CIFAR-10

You now have the foundation to build and train your own neural network architectures!